# No-Cloning Theorem

The **no-cloning theorem** states that it is impossible to create an
exact copy of an arbitrary unknown quantum state.

## Statement

There is no unitary operation $U$ such that for **every** single-qubit
state $|\psi\rangle$:

$$U\,|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$$

## Why it matters

- Makes quantum eavesdropping detectable (BB84, B92 protocols).
- Rules out perfect quantum error correction by simple copying.
- Explains why quantum teleportation *destroys* the source state.

## Proof by contradiction (linearity argument)

Suppose a cloner $C$ works for the two basis states:

$$C\,|0\rangle = |0\rangle|0\rangle, \qquad C\,|1\rangle = |1\rangle|1\rangle$$

Now apply $C$ to the superposition
$|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$.

**What cloning demands** (copy the whole state):

$$C\,|+\rangle\stackrel{?}{=}|+\rangle|+\rangle
  = \frac{1}{2}\bigl(|00\rangle+|01\rangle+|10\rangle+|11\rangle\bigr)$$

**What linearity gives** (distribute $C$ over the sum):

$$C\,|+\rangle = \frac{1}{\sqrt{2}}\bigl(C|0\rangle+C|1\rangle\bigr)
  = \frac{1}{\sqrt{2}}\bigl(|00\rangle+|11\rangle\bigr)$$

These two results are **different** -- the first has four terms, the
second only two (a Bell state).  The contradiction proves no such $C$
can exist for all states.

## Numeric check

Let $|+\rangle = (|0\rangle+|1\rangle)/\sqrt{2}$.

- $|+\rangle|+\rangle = \frac{1}{2}(|00\rangle+|01\rangle+|10\rangle+|11\rangle)$
- $\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle) = \frac{1}{\sqrt{2}}|00\rangle + \frac{1}{\sqrt{2}}|11\rangle$

The amplitudes of $|01\rangle$ and $|10\rangle$ are $1/2$ in the first
expression and $0$ in the second.  They cannot be equal.

## Imports

From *Mathematics of Quantum Computing*, section 4.6.2
"Perfect Quantum Copier".

In [1]:
from sympy.physics.quantum.qubit import Qubit, matrix_to_qubit
from sympy import sqrt
from sympy.physics.quantum import TensorProduct
from sympy.physics.quantum.represent import represent

## The hypothetical cloner

We define a function `clone(q, blank)` that models the *desired*
behaviour of a perfect cloner: it ignores the blank ancilla qubit and
simply returns $|q\rangle|q\rangle$.

If such a function were physically realizable it would have to be
linear (all quantum operations are linear).  We will show that
linearity and perfect cloning are incompatible.

In [2]:
def clone(state, blank):
    """Model a perfect cloner: return |state>|state>.

    Parameters
    ----------
    state : quantum state to clone
    blank : ancilla qubit (ignored; any value is accepted)

    Returns
    -------
    TensorProduct of state with itself.

    Note: this function cannot correspond to a physical unitary
    because it violates linearity for superposition inputs.
    """
    return TensorProduct(state, state)

## Step 1 -- clone $|0\rangle$

A valid cloner must satisfy $C|0\rangle|\text{blank}\rangle = |0\rangle|0\rangle$.

$$\text{clone}(|0\rangle) = |0\rangle \otimes |0\rangle = |00\rangle$$

In [3]:
# clone |0>: result must be |00>
blank = Qubit(0)   # ancilla, content does not matter
result_0 = clone(Qubit(0), blank)
print(f"clone(|0>) = {result_0}")
assert result_0 == TensorProduct(Qubit(0), Qubit(0))

## Step 2 -- clone $|1\rangle$

$$\text{clone}(|1\rangle) = |1\rangle \otimes |1\rangle = |11\rangle$$

In [4]:
# clone |1>: result must be |11>
result_1 = clone(Qubit(1), blank)
print(f"clone(|1>) = {result_1}")
assert result_1 == TensorProduct(Qubit(1), Qubit(1))

## Step 3 -- clone the superposition $|+\rangle$

Now we apply the cloner to
$|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$.

The *desired* output is a perfect copy:

$$|+\rangle \otimes |+\rangle
  = \frac{1}{2}\bigl(|00\rangle+|01\rangle+|10\rangle+|11\rangle\bigr)$$

This has four equal-amplitude terms.

In [5]:
# |+> = (|0> + |1>) / sqrt(2)
plus = 1 / sqrt(2) * (Qubit(0) + Qubit(1))

# desired: clone(|+>) = |+>|+> (four-term product state)
desired = clone(plus, blank)
desired = matrix_to_qubit(represent(desired))
print(f"desired  clone(|+>) = {desired}")

## Step 4 -- what linearity actually gives

Any physical (unitary) operation must be **linear**.  Distributing $C$
over the superposition:

$$C\,|+\rangle
  = \frac{1}{\sqrt{2}}\bigl(C|0\rangle + C|1\rangle\bigr)
  = \frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$$

This is a **Bell state** -- only two terms, no $|01\rangle$ or
$|10\rangle$ components.

In [6]:
# linear: distribute clone over the superposition
# C(|+>) = 1/sqrt(2) * (C|0> + C|1>)
linear_result = 1 / sqrt(2) * (
    clone(Qubit(0), blank) + clone(Qubit(1), blank)
)
linear_result = matrix_to_qubit(represent(linear_result))
print(f"linear   clone(|+>) = {linear_result}")

## Step 5 -- contradiction

The two expressions are not equal:

Amplitude comparison by basis state:

- $|00\rangle$: desired $= 1/2$, linear $= 1/\sqrt{2}$
- $|01\rangle$: desired $= 1/2$, linear $= 0$
- $|10\rangle$: desired $= 1/2$, linear $= 0$
- $|11\rangle$: desired $= 1/2$, linear $= 1/\sqrt{2}$

The $|01\rangle$ and $|10\rangle$ amplitudes differ, so no linear
(hence no unitary) operation can clone all states.  **QED.**

In [7]:
# the two results must differ -- this is the no-cloning contradiction
print(f"desired       = {desired}")
print(f"linear result = {linear_result}")
print(f"Are they equal? {desired == linear_result}")
assert desired != linear_result, (
    "Cloning a superposition is inconsistent with linearity"
)